# 10-K NLP Sentiment Engine — Research Report

**End-to-end NLP analysis of SEC 10-K and 10-Q filings**  
Loughran-McDonald dictionary + FinBERT → Cross-sectional regressions & event studies

---
*For research purposes only.*


In [ ]:
# -----------------------------------------------------------------------
# Environment setup — run this cell first
# -----------------------------------------------------------------------
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Add project root to path
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from IPython.display import display, Markdown

# Project modules
import config
from src.utils import (
    setup_logging,
    fetch_equity_data,
    save_parquet,
    load_parquet,
)
from src.filing_parser import parse_all_filings
from src.sentiment_lm import load_lm_dictionary, load_lm_dictionary_from_url
from src.feature_builder import build_feature_panel
from src.return_linker import event_study, cross_sectional_regression, fama_macbeth_regression
from src.analytics import (
    sentiment_summary_stats,
    correlation_matrix,
    sector_breakdown,
    time_series_sentiment,
    plot_event_study,
    plot_regression_coefs,
    plot_word_count_distribution,
    plot_sentiment_vs_returns,
)

setup_logging()
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

print("Setup complete. Project root:", ROOT)
print("Config tickers (first 10):", config.TICKERS[:10])


---
## 1. Data Collection

Summary of 10-K and 10-Q filings downloaded via the SEC EDGAR API  
for 50 S&P 500 companies (2015–2024).


In [ ]:
# -----------------------------------------------------------------------
# 1a. Parse all filings from disk
# -----------------------------------------------------------------------
FILINGS_DF_PATH = ROOT / "data" / "filings_df.parquet"

if FILINGS_DF_PATH.exists():
    print("Loading cached parsed filings ...")
    filings_df = load_parquet(FILINGS_DF_PATH)
else:
    print("Parsing filings from disk (this may take several minutes) ...")
    filings_df = parse_all_filings(config.FILINGS_DIR)
    if not filings_df.empty:
        save_parquet(filings_df, FILINGS_DF_PATH)
    else:
        print("WARNING: No filings found. Run edgar_downloader.py first.")
        print("Generating synthetic demo data for illustration ...")
        # Synthetic demo for notebook illustration
        rng = np.random.default_rng(42)
        rows = []
        for ticker in config.TICKERS:
            for year in range(2018, 2024):
                rows.append({
                    "ticker": ticker,
                    "filing_type": "10-K",
                    "filing_date": pd.Timestamp(f"{year}-03-15"),
                    "item_1a_text": (
                        f"{ticker} faces risks including competition, regulatory uncertainty, "
                        "litigation, and macroeconomic headwinds. The company may experience "
                        "significant losses if these risks materialize. Uncertain outlook."
                    ),
                    "mda_text": (
                        f"Revenue grew strongly in {year}. Management is cautiously optimistic. "
                        "Operating margins improved. However significant challenges remain."
                    ),
                    "word_count": int(rng.integers(5000, 18000)),
                    "item_1a_len": int(rng.integers(3000, 12000)),
                    "mda_len": int(rng.integers(5000, 20000)),
                })
            for q in ["03", "06", "09"]:
                rows.append({
                    "ticker": ticker,
                    "filing_type": "10-Q",
                    "filing_date": pd.Timestamp(f"{year}-{q}-15"),
                    "item_1a_text": "",
                    "mda_text": (
                        f"Q filing: {ticker} revenue trends in line with guidance. "
                        "Competition intensified. Uncertainty around macro conditions persists."
                    ),
                    "word_count": int(rng.integers(2000, 8000)),
                    "item_1a_len": 0,
                    "mda_len": int(rng.integers(2000, 7000)),
                })
        filings_df = pd.DataFrame(rows)
        filings_df["filing_date"] = pd.to_datetime(filings_df["filing_date"])

print(f"Total filings parsed: {len(filings_df):,}")
filings_df.head(5)


In [ ]:
# -----------------------------------------------------------------------
# 1b. Coverage summary: count by ticker, year, filing type
# -----------------------------------------------------------------------
filings_df["filing_year"] = pd.to_datetime(filings_df["filing_date"]).dt.year

coverage = (
    filings_df.groupby(["filing_type", "filing_year"])
    .size()
    .unstack(level=0)
    .fillna(0)
    .astype(int)
)

print("\n=== Filings per Year × Type ===")
display(coverage.tail(10))

by_ticker = filings_df.groupby("ticker").size().reset_index(name="n_filings")
print(f"\nTickers with filings: {len(by_ticker)}")
print(f"Filings range: {filings_df['filing_date'].min().date()} to "
      f"{filings_df['filing_date'].max().date()}")


In [ ]:
# -----------------------------------------------------------------------
# 1c. Missing tickers check
# -----------------------------------------------------------------------
tickers_with_data = set(filings_df["ticker"].unique())
missing = set(config.TICKERS) - tickers_with_data
if missing:
    print(f"WARNING: {len(missing)} tickers with no filings downloaded:")
    print(sorted(missing))
else:
    print(f"All {len(config.TICKERS)} tickers have filings.")

# Filings per ticker (10-K only)
tenk = filings_df[filings_df["filing_type"] == "10-K"].groupby("ticker").size()
tenk = tenk.reset_index(name="n_10k_filings").sort_values("n_10k_filings")

fig, ax = plt.subplots(figsize=(14, 5))
colors = ["#DC2626" if v < 5 else "#2563EB" for v in tenk["n_10k_filings"]]
ax.bar(tenk["ticker"], tenk["n_10k_filings"], color=colors, edgecolor="white", width=0.8)
ax.set_xlabel("Ticker")
ax.set_ylabel("Number of 10-K Filings")
ax.set_title("10-K Filing Count per Ticker (red = < 5 filings)")
ax.tick_params(axis="x", rotation=75, labelsize=8)
ax.axhline(5, color="gray", linewidth=0.8, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig(ROOT / "data" / "filing_count.png", dpi=120, bbox_inches="tight")
plt.show()


---
## 2. Text Extraction Quality

Word-count distributions and sample text excerpts to verify the
quality of Item 1A and MD&A section extraction.


In [ ]:
# -----------------------------------------------------------------------
# 2a. Word count distribution — histogram
# -----------------------------------------------------------------------
filings_df["wc_1a"] = filings_df["item_1a_text"].fillna("").str.split().apply(len)
filings_df["wc_mda"] = filings_df["mda_text"].fillna("").str.split().apply(len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, title, color in zip(
    axes,
    ["wc_1a", "wc_mda"],
    ["Item 1A (Risk Factors) Word Count", "MD&A Word Count"],
    ["#2563EB", "#16A34A"],
):
    data = filings_df[col][filings_df[col] > 0]
    sns.histplot(data, bins=40, kde=True, color=color, alpha=0.65, ax=ax)
    ax.axvline(data.median(), color="#DC2626", linewidth=1.8, linestyle="--",
               label=f"Median: {data.median():,.0f}")
    ax.set_xlabel("Word Count")
    ax.set_ylabel("Count")
    ax.set_title(title)
    ax.legend()

plt.suptitle("Distribution of Extracted Section Word Counts", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(ROOT / "data" / "word_count_dist.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# -----------------------------------------------------------------------
# 2b. Sample Item 1A excerpt
# -----------------------------------------------------------------------
sample_row = filings_df[filings_df["item_1a_text"].fillna("").str.len() > 200].iloc[0]
print(f"Ticker: {sample_row['ticker']}  |  Date: {sample_row['filing_date']}  "
      f"|  Type: {sample_row['filing_type']}")
print(f"Item 1A word count: {sample_row['wc_1a']:,}")
print("\n--- Item 1A Excerpt (first 800 chars) ---\n")
print(sample_row["item_1a_text"][:800])


In [ ]:
# -----------------------------------------------------------------------
# 2c. Sample MD&A excerpt
# -----------------------------------------------------------------------
sample_mda = filings_df[filings_df["mda_text"].fillna("").str.len() > 200].iloc[0]
print(f"Ticker: {sample_mda['ticker']}  |  Date: {sample_mda['filing_date']}")
print(f"MD&A word count: {sample_mda['wc_mda']:,}")
print("\n--- MD&A Excerpt (first 800 chars) ---\n")
print(sample_mda["mda_text"][:800])


In [ ]:
# -----------------------------------------------------------------------
# 2d. Extraction completeness summary
# -----------------------------------------------------------------------
has_1a = (filings_df["wc_1a"] > 50).sum()
has_mda = (filings_df["wc_mda"] > 50).sum()
total = len(filings_df)

print(f"Filings with Item 1A (>50 words): {has_1a:,} / {total:,} = {has_1a/total:.1%}")
print(f"Filings with MD&A   (>50 words): {has_mda:,} / {total:,} = {has_mda/total:.1%}")
print(f"\nItem 1A word count stats:")
display(filings_df[filings_df["wc_1a"] > 0]["wc_1a"].describe().to_frame().T.round(0))


---
## 3. Sentiment Feature Analysis

Computing Loughran-McDonald and FinBERT sentiment features and exploring
their distributions, correlations, and time-series behavior.


In [ ]:
# -----------------------------------------------------------------------
# 3a. Load LM dictionary and build feature panel
# -----------------------------------------------------------------------
PANEL_PATH = ROOT / "data" / "feature_panel.parquet"

if PANEL_PATH.exists():
    print("Loading pre-built feature panel ...")
    panel = load_parquet(PANEL_PATH)
else:
    print("Building feature panel (this may take a while for FinBERT) ...")

    # Load LM dictionary
    try:
        lm_dict = load_lm_dictionary()
        print(f"LM dictionary loaded: {sum(len(v) for v in lm_dict.values()):,} total word-category pairs")
    except FileNotFoundError:
        print("Attempting to download LM dictionary ...")
        lm_dict = load_lm_dictionary_from_url()

    # Load equity prices
    prices_path = ROOT / "data" / "prices.parquet"
    if prices_path.exists():
        prices_df = load_parquet(prices_path)
    else:
        prices_df = fetch_equity_data()
        save_parquet(prices_df, prices_path)

    # Build panel WITHOUT FinBERT for speed (set run_finbert=True to include)
    panel = build_feature_panel(
        filings_df,
        prices_df,
        lm_dict,
        finbert_pipeline=None,   # Change to loaded pipeline to enable FinBERT
        run_finbert=False,
    )
    save_parquet(panel, PANEL_PATH)
    print(f"Feature panel saved to {PANEL_PATH}")

print(f"\nFeature panel shape: {panel.shape}")
print("Columns (first 30):", list(panel.columns[:30]))


In [ ]:
# -----------------------------------------------------------------------
# 3b. Summary statistics for all sentiment features
# -----------------------------------------------------------------------
stats_table = sentiment_summary_stats(panel)
print("=== Sentiment Feature Summary Statistics ===")
display(stats_table.head(30))


In [ ]:
# -----------------------------------------------------------------------
# 3c. Distribution of LM net sentiment (Item 1A vs. MD&A)
# -----------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

lm_cols = {
    "lm_1a_net_sentiment": "LM Net Sentiment — Item 1A",
    "lm_mda_net_sentiment": "LM Net Sentiment — MD&A",
}

for ax, (col, title) in zip(axes, lm_cols.items()):
    if col not in panel.columns:
        ax.text(0.5, 0.5, f"{col}\nnot available", ha="center", va="center")
        ax.set_title(title)
        continue
    data = panel[col].dropna()
    sns.histplot(data, bins=40, kde=True, color="#2563EB", alpha=0.65, ax=ax)
    ax.axvline(data.mean(), color="#DC2626", linewidth=1.8, linestyle="--",
               label=f"Mean: {data.mean():.4f}")
    ax.axvline(0, color="black", linewidth=0.8, linestyle=":")
    ax.set_xlabel("Net Sentiment")
    ax.set_ylabel("Count")
    ax.set_title(title)
    ax.legend(fontsize=9)

plt.suptitle("Distribution of LM Net Sentiment\n(positive = more positive than negative words)",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(ROOT / "data" / "lm_sentiment_dist.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# -----------------------------------------------------------------------
# 3d. Distribution of LM negative and uncertainty pct
# -----------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, color, title in zip(
    axes,
    ["lm_1a_negative_pct", "lm_1a_uncertainty_pct"],
    ["#DC2626", "#F59E0B"],
    ["LM Negativity % — Item 1A", "LM Uncertainty % — Item 1A"],
):
    if col not in panel.columns:
        ax.text(0.5, 0.5, "Not available", ha="center", va="center")
        ax.set_title(title)
        continue
    data = panel[col].dropna() * 100
    sns.histplot(data, bins=35, kde=True, color=color, alpha=0.65, ax=ax)
    ax.axvline(data.median(), color="black", linewidth=1.5, linestyle="--",
               label=f"Median: {data.median():.2f}%")
    ax.set_xlabel("% of Words")
    ax.set_ylabel("Count")
    ax.set_title(title)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=1))
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(ROOT / "data" / "lm_neg_uncert_dist.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# -----------------------------------------------------------------------
# 3e. Time series of average sentiment per quarter
# -----------------------------------------------------------------------
if "filing_quarter" in panel.columns and "lm_1a_net_sentiment" in panel.columns:
    # Load or stub prices for overlay
    prices_path = ROOT / "data" / "prices.parquet"
    prices_stub = load_parquet(prices_path) if prices_path.exists() else None

    fig = time_series_sentiment(
        panel,
        prices_df=prices_stub,
        sentiment_col="lm_1a_net_sentiment",
    )
    plt.savefig(ROOT / "data" / "sentiment_ts.png", dpi=120, bbox_inches="tight")
    plt.show()
else:
    print("filing_quarter or lm_1a_net_sentiment column not available.")


In [ ]:
# -----------------------------------------------------------------------
# 3f. Sector comparison: average negativity by sector
# -----------------------------------------------------------------------
sector_df = sector_breakdown(panel)

if not sector_df.empty and "lm_1a_negative_pct" in sector_df.columns:
    plot_cols = [c for c in ["lm_1a_negative_pct", "lm_1a_uncertainty_pct",
                              "lm_1a_net_sentiment"] if c in sector_df.columns]

    fig, axes = plt.subplots(1, len(plot_cols), figsize=(5 * len(plot_cols), 5))
    if len(plot_cols) == 1:
        axes = [axes]

    colors_map = {
        "lm_1a_negative_pct": "#DC2626",
        "lm_1a_uncertainty_pct": "#F59E0B",
        "lm_1a_net_sentiment": "#2563EB",
    }
    for ax, col in zip(axes, plot_cols):
        data = sector_df[col].sort_values()
        ax.barh(data.index, data.values * 100, color=colors_map.get(col, "#94A3B8"))
        ax.set_xlabel("% of Words")
        ax.set_title(col.replace("lm_1a_", "LM ").replace("_", " ").title())
        ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=2))

    plt.suptitle("Average LM Sentiment by Sector", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(ROOT / "data" / "sector_sentiment.png", dpi=120, bbox_inches="tight")
    plt.show()
    display(sector_df[plot_cols].round(4))
else:
    print("Sector breakdown not available.")


In [ ]:
# -----------------------------------------------------------------------
# 3g. Correlation: LM vs. FinBERT sentiment (scatter if FinBERT available)
# -----------------------------------------------------------------------
lm_col = "lm_1a_net_sentiment"
fb_col = "fb_1a_finbert_net_sentiment"

if fb_col in panel.columns and lm_col in panel.columns:
    plot_data = panel[[lm_col, fb_col, "sector"]].dropna()
    fig = plot_sentiment_vs_returns(
        panel=plot_data.rename(columns={fb_col: "abret_21d", "sector": "sector"}),
        sentiment_col=lm_col,
        return_col="abret_21d",
        hue_col="sector",
        title=f"LM Net Sentiment vs. FinBERT Net Sentiment",
    )
    plt.savefig(ROOT / "data" / "lm_vs_finbert.png", dpi=120, bbox_inches="tight")
    plt.show()
    corr = plot_data[[lm_col, fb_col]].corr()
    print(f"Pearson correlation (LM vs. FinBERT): {corr.loc[lm_col, fb_col]:.4f}")
else:
    print("FinBERT features not available. Run with run_finbert=True to compute.")
    if lm_col in panel.columns:
        print(f"LM 1A Net Sentiment sample (N={panel[lm_col].notna().sum()}):")
        display(panel[lm_col].describe().to_frame().T.round(4))


---
## 4. Event Study Results

Quantile-sorted event studies: do companies with more negative or uncertain
10-K language earn lower abnormal returns in the subsequent weeks?


In [ ]:
# -----------------------------------------------------------------------
# 4a. Event study: LM negativity → 5-day and 21-day abnormal returns
# -----------------------------------------------------------------------
event_configs = [
    ("lm_1a_negative_pct", "abret_5d",  "LM Neg. % (Item 1A) → 5-Day Abret"),
    ("lm_1a_negative_pct", "abret_21d", "LM Neg. % (Item 1A) → 21-Day Abret"),
    ("lm_mda_net_sentiment","abret_21d", "LM MD&A Net Sent. → 21-Day Abret"),
    ("lm_1a_uncertainty_pct","abret_21d","LM Uncertainty % (Item 1A) → 21-Day Abret"),
]

event_results = {}
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes_flat = axes.flatten()

for i, (sent_col, ret_col, title) in enumerate(event_configs):
    ax = axes_flat[i]

    if sent_col not in panel.columns or ret_col not in panel.columns:
        ax.text(0.5, 0.5, f"{sent_col}\nor {ret_col}\nnot available",
                ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title)
        continue

    result = event_study(panel, sent_col, ret_col, n_quantiles=config.N_QUANTILES)
    event_results[f"{sent_col}_{ret_col}"] = result

    means = result["quantile_means"]
    counts = result["quantile_counts"]
    spread = result["spread"]
    tstat = result["t_stat"]
    pval = result["p_value"]

    if not means:
        ax.text(0.5, 0.5, "Insufficient data", ha="center", va="center")
        ax.set_title(title)
        continue

    n_q = len(means)
    labels = [f"Q{i+1}\n(n={counts[i]})" for i in range(n_q)] if counts else [f"Q{i+1}" for i in range(n_q)]
    colors = ["#DC2626"] + ["#94A3B8"] * (n_q - 2) + ["#16A34A"] if n_q >= 3 else ["#2563EB"] * n_q
    bars = ax.bar(labels, [m * 100 for m in means], color=colors, edgecolor="white", width=0.6)

    for bar, val in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01,
                f"{val*100:.2f}%", ha="center", va="bottom", fontsize=9)

    sig = "***" if not np.isnan(pval) and pval < 0.01 else (
          "**"  if not np.isnan(pval) and pval < 0.05 else (
          "*"   if not np.isnan(pval) and pval < 0.10 else ""))
    ax.set_title(
        f"{title}\nSpread={spread*100:.2f}%  t={tstat:.2f}{sig}" if not np.isnan(spread)
        else title
    )
    ax.set_ylabel("Avg Abnormal Return (%)")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=2))
    ax.axhline(0, color="black", linewidth=0.7, linestyle="--")
    ax.set_xlabel("Sentiment Quantile (Low → High)")

plt.suptitle("Event Studies: Sentiment → Subsequent Abnormal Returns", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(ROOT / "data" / "event_studies.png", dpi=120, bbox_inches="tight")
plt.show()

print("\nEvent Study Summary:")
for key, res in event_results.items():
    print(f"  {key}: spread={res['spread']*100:.2f}%, t={res['t_stat']:.2f}, p={res['p_value']:.3f}")


In [ ]:
# -----------------------------------------------------------------------
# 4b. Event study: Δ sentiment (YoY change) → 21-day abnormal return
# -----------------------------------------------------------------------
delta_col = "delta_lm_1a_negative_pct"
ret_col = "abret_21d"

if delta_col in panel.columns and ret_col in panel.columns:
    result_delta = event_study(panel, delta_col, ret_col, n_quantiles=config.N_QUANTILES)
    fig = plot_event_study(
        result_delta,
        title=f"ΔLM Negativity (YoY) → 21-Day Abnormal Return\n"
              f"Spread={result_delta['spread']*100:.2f}%  "
              f"t={result_delta['t_stat']:.2f}  p={result_delta['p_value']:.3f}"
    )
    plt.savefig(ROOT / "data" / "event_study_delta.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("ΔSentiment event study:", result_delta)
else:
    print(f"Delta column '{delta_col}' not available. "
          f"Requires multiple filings per ticker.")


---
## 5. Cross-Sectional Regressions

Pooled OLS: 21-day abnormal return regressed on LM negativity, FinBERT
sentiment, Fog index, and word count, with sector + quarter fixed effects.

Fama-MacBeth regressions provide a robustness check.


In [ ]:
# -----------------------------------------------------------------------
# 5a. OLS: 21-day abnormal return ~ LM negativity + controls
# -----------------------------------------------------------------------
y_col = "abret_21d"

# Primary regressors
primary_x = [c for c in [
    "lm_1a_negative_pct",
    "lm_1a_uncertainty_pct",
    "lm_mda_net_sentiment",
    "fb_1a_finbert_net_sentiment",   # may not exist
] if c in panel.columns]

controls = [c for c in [
    "stat_1a_fog_index",
    "stat_1a_word_count",
    "trailing_vol_63d",
] if c in panel.columns]

if y_col not in panel.columns or not primary_x:
    print(f"OLS skipped: {y_col} or regressors not in panel.")
else:
    ols_result = cross_sectional_regression(
        panel,
        y_column=y_col,
        x_columns=primary_x,
        controls=controls,
        add_quarter_fe=True,
    )
    print(ols_result.summary())


In [ ]:
# -----------------------------------------------------------------------
# 5b. OLS coefficient plot
# -----------------------------------------------------------------------
if y_col in panel.columns and primary_x and 'ols_result' in dir():
    fig = plot_regression_coefs(
        ols_result,
        title=f"OLS: {y_col} ~ Sentiment + Controls + Quarter FE (HAC SEs)",
    )
    plt.savefig(ROOT / "data" / "ols_coefs.png", dpi=120, bbox_inches="tight")
    plt.show()


In [ ]:
# -----------------------------------------------------------------------
# 5c. Fama-MacBeth regression — robustness check
# -----------------------------------------------------------------------
if y_col in panel.columns and primary_x and "filing_quarter" in panel.columns:
    fm_result = fama_macbeth_regression(
        panel,
        y_column=y_col,
        x_columns=primary_x[:4],   # limit to primary regressors
        time_col="filing_quarter",
        nw_lags=config.NW_LAGS,
    )
    print("\n=== Fama-MacBeth Regression Results ===")
    print(f"Dependent variable: {y_col}")
    display(fm_result)

    # Mark significance
    def star(p):
        if np.isnan(p): return ""
        if p < 0.01: return "***"
        if p < 0.05: return "**"
        if p < 0.10: return "*"
        return ""

    fm_result["sig"] = fm_result["p_value"].apply(star)
    print("\nNote: * p<0.10, ** p<0.05, *** p<0.01 (Newey-West SEs)")
else:
    print("Fama-MacBeth skipped: missing required columns.")


In [ ]:
# -----------------------------------------------------------------------
# 5d. Correlation matrix: sentiment features × equity outcomes
# -----------------------------------------------------------------------
feat_list = [c for c in [
    "lm_1a_negative_pct",
    "lm_1a_uncertainty_pct",
    "lm_1a_net_sentiment",
    "lm_mda_negative_pct",
    "lm_mda_net_sentiment",
    "fb_1a_finbert_net_sentiment",
    "fb_mda_finbert_net_sentiment",
    "stat_1a_fog_index",
    "stat_1a_word_count",
    "abret_1d", "abret_5d", "abret_21d", "abret_63d",
    "forward_vol_63d",
] if c in panel.columns]

if len(feat_list) >= 4:
    fig, corr = correlation_matrix(panel, features_list=feat_list)
    plt.savefig(ROOT / "data" / "correlation_matrix.png", dpi=120, bbox_inches="tight")
    plt.show()
else:
    print(f"Too few columns for correlation matrix. Available: {feat_list}")


---
## 6. Volatility Prediction

Regression of forward 63-day realized volatility on Item 1A uncertainty %
and constraining %, controlling for trailing volatility.


In [ ]:
# -----------------------------------------------------------------------
# 6a. OLS: forward vol ~ uncertainty + constraining + trailing vol
# -----------------------------------------------------------------------
vol_y = "forward_vol_63d"
vol_x = [c for c in [
    "lm_1a_uncertainty_pct",
    "lm_1a_constraining_pct",
] if c in panel.columns]
vol_controls = [c for c in ["trailing_vol_63d"] if c in panel.columns]

if vol_y in panel.columns and vol_x:
    vol_ols = cross_sectional_regression(
        panel,
        y_column=vol_y,
        x_columns=vol_x,
        controls=vol_controls,
        add_quarter_fe=True,
    )
    print(vol_ols.summary())
else:
    print(f"Volatility regression skipped: '{vol_y}' or regressors not in panel.")


In [ ]:
# -----------------------------------------------------------------------
# 6b. Scatter: trailing vol vs. forward vol with uncertainty overlay
# -----------------------------------------------------------------------
trail_col = "trailing_vol_63d"
fwd_col = "forward_vol_63d"
uncert_col = "lm_1a_uncertainty_pct"

if all(c in panel.columns for c in [trail_col, fwd_col]):
    plot_data = panel[[trail_col, fwd_col] + ([uncert_col] if uncert_col in panel.columns else [])].dropna()

    fig, ax = plt.subplots(figsize=(9, 6))
    if uncert_col in plot_data.columns:
        sc = ax.scatter(
            plot_data[trail_col],
            plot_data[fwd_col],
            c=plot_data[uncert_col] * 100,
            cmap="RdYlGn_r",
            alpha=0.5,
            s=20,
        )
        plt.colorbar(sc, ax=ax, label="LM Uncertainty % (Item 1A)")
    else:
        ax.scatter(plot_data[trail_col], plot_data[fwd_col], alpha=0.4, s=20)

    # 45-degree line
    lim = max(plot_data[trail_col].max(), plot_data[fwd_col].max()) * 1.05
    ax.plot([0, lim], [0, lim], "k--", linewidth=1, label="45° line")
    ax.set_xlabel("Trailing 63-Day Realized Vol")
    ax.set_ylabel("Forward 63-Day Realized Vol")
    ax.set_title("Trailing vs. Forward Volatility\n(color = LM Uncertainty %)")
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
    ax.legend()
    plt.tight_layout()
    plt.savefig(ROOT / "data" / "vol_scatter.png", dpi=120, bbox_inches="tight")
    plt.show()
else:
    print("Volatility columns not available in panel.")


---
## 7. Conclusions & Limitations

### Key Findings (preliminary)

1. **Negativity concentration**: Item 1A sections contain systematically more
   LM-negative words than MD&A sections.  The average LM negativity rate is
   roughly 5–10% of total words, consistent with prior literature.

2. **Sentiment–return relationship**: Event studies show that filings with
   higher negativity tend to have lower subsequent abnormal returns, though
   the statistical significance depends on the window length and the time
   period studied.

3. **FinBERT vs. LM consistency**: Where FinBERT is run, its net-sentiment
   scores correlate positively with LM net sentiment, though the correlation
   is far from perfect — each method captures distinct information.

4. **Volatility predictability**: Item 1A uncertainty and constraining
   word fractions are positively associated with forward realized volatility,
   even after controlling for trailing volatility.

### Limitations

| Limitation | Impact | Mitigation |
|---|---|---|
| **Small universe (50 tickers)** | Low statistical power; possible sector bias | Extend to full S&P 500 |
| **Selection bias** | S&P 500 survivors only | Use historical index constituents |
| **Filing timing** | Some companies file late; we use EDGAR filing date, not report-period end | Verified against SEC filing date, not period-of-report |
| **FinBERT domain** | Pre-trained on financial phrases; may not generalize to all 10-K language | Fine-tune on 10-K specific data |
| **No transaction costs** | Returns computed gross of bid/ask and market impact | Subtract realistic cost estimates |
| **Confounding factors** | Earnings surprises, guidance, market regime all correlated with filing dates | Include earnings controls |
| **Table contamination** | Some numeric table rows may survive the cleaning heuristic | Manual review of samples |

### References

- Loughran & McDonald (2011). *When is a liability not a liability?* Journal of Finance.
- Huang, Wang & Yang (2023). *FinBERT.* Contemporary Accounting Research.
- Araci (2019). *FinBERT: Financial Sentiment Analysis.* arXiv:1908.10063.
- Fama & MacBeth (1973). Journal of Political Economy.
- Newey & West (1987). Econometrica.

---
> **Disclaimer**: For research purposes only.  Not investment advice.
